# Federal Reserve Press Releases — LLM Relevance Audit (200 records)

เป้าหมายของ Notebook นี้คือ:
1. สุ่ม 200 records จาก Clean Master ด้วย stratified sampling ตาม `year × document_type`
2. ส่งเฉพาะ sample 200 records ให้ LLM ช่วยจำแนกความเกี่ยวข้องกับ Gold Assistant
3. บันทึก label, confidence, evidence, factors และเหตุผล
4. สร้าง CSV ให้มนุษย์ตรวจผล LLM และเก็บ sampling metadata

**ไม่แก้/ลบ Clean Master** — อ่าน input แบบ read-only และเขียนผลลงโฟลเดอร์ใหม่

> Notebook นี้ใช้ **ThaiLLM** (`openthaigpt-thaillm-8b-instruct-v7.2`) ผ่าน endpoint แบบ OpenAI-compatible
> และกระจายงานไปยัง API key ทั้ง 7 ตัว (`THAILLM_API_KEY1..7` ที่โหลดจาก `.env`) แบบขนาน (parallel)
>
> ควรตรวจนโยบายข้อมูล/ค่าใช้จ่ายก่อนส่งเนื้อหาออกจากเครื่อง


In [12]:
from pathlib import Path
import os, json, re, time, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
import pandas as pd
import numpy as np
from IPython.display import display
from tqdm.auto import tqdm

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


pandas: 3.0.6
numpy: 2.4.6


In [13]:
# ===== CONFIG =====
# แก้ path นี้ให้ตรงกับ Clean Master ของคุณ
INPUT_PATH = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl")
OUTPUT_DIR = Path(r"D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned_audit")

SAMPLE_SIZE = 200
RANDOM_SEED = 20261026

# ===== ThaiLLM API config (OpenAI-compatible endpoint) =====
from dotenv import load_dotenv
load_dotenv()

THAILLM_BASE_URL = "http://thaillm.or.th/api/v1"
THAILLM_MODEL = "openthaigpt-thaillm-8b-instruct-v7.2"

# โหลด 7 API keys จาก .env (THAILLM_API_KEY1 .. THAILLM_API_KEY7)
THAILLM_API_KEYS = [os.getenv(f"THAILLM_API_KEY{i}") for i in range(1, 8)]
THAILLM_API_KEYS = [key for key in THAILLM_API_KEYS if key]

if not THAILLM_API_KEYS:
    raise EnvironmentError(
        "ไม่พบ THAILLM_API_KEY1..7 ใน .env — ตั้งค่า API key ใน файл .env ก่อนรัน "
        "อย่าเขียน API key ลงใน Notebook หรือ CSV"
    )

# ThaiLLM rate limit: 5 req/s, 200 req/min ต่อ 1 key → 7 keys ≈ 35 req/s
# delay = 1/5 = 0.2s ใช้ 0.25s เผื่อ safety margin (ต่อ worker 1 key)
THAILLM_DELAY = 0.25

PROVIDER = "ThaiLLM"
MODEL = THAILLM_MODEL
TEMPERATURE = 0
MAX_TEXT_CHARS = 12000
MAX_TOKENS = 1500
MAX_RETRIES = 4

TYPE_FIELDS = ["document_type", "release_type", "category", "type", "page_type"]
DATE_FIELDS = ["published_at", "publication_date", "date", "release_date"]
TITLE_FIELDS = ["title", "headline", "name"]
URL_FIELDS = ["url", "source_url", "link"]
TEXT_FIELDS = ["clean_text", "body", "content", "text", "article_text"]

LABELS = ["DIRECT_GOLD", "MACRO_RELEVANT", "LOW_RELEVANCE", "UNCERTAIN"]

print("Input:", INPUT_PATH.resolve())
print("Output:", OUTPUT_DIR.resolve())
print("Provider:", PROVIDER, "| Model:", MODEL)
print("API keys loaded:", len(THAILLM_API_KEYS), "| delay/key:", THAILLM_DELAY, "s")
print("Seed:", RANDOM_SEED, "| Sample:", SAMPLE_SIZE)


Input: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned\fed_press_releases_clean.jsonl
Output: D:\Project_Github\gold assisstant\dataset\news\fed\fed_press_releases_cleaned_audit
Provider: ThaiLLM | Model: openthaigpt-thaillm-8b-instruct-v7.2
API keys loaded: 7 | delay/key: 0.25 s
Seed: 20261026 | Sample: 200


In [14]:
# ===== LOAD CLEAN MASTER READ-ONLY =====
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"ไม่พบ Clean Master: {INPUT_PATH} — แก้ INPUT_PATH ใน Cell 2")

records = []
with INPUT_PATH.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if not line.strip():
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"JSONL parse error ที่บรรทัด {line_no}: {exc}") from exc
        if isinstance(obj, dict):
            records.append(obj)

if not records:
    raise ValueError("ไม่พบ JSON object ใน Clean Master")

df = pd.DataFrame(records)
print(f"Loaded {len(df):,} records")
display(pd.DataFrame({"column": df.columns, "non_null": [df[c].notna().sum() for c in df.columns]}))

Loaded 4,629 records


,column,non_null
0,document_id,4629
1,source,4629
2,title,4629
3,publication_date,4629
4,publication_date_source,4629
5,release_time_text,4629
6,document_type,4629
7,source_archive_year,4629
8,url,4629
9,content,4629


In [15]:
# ===== PREPARE YEAR × DOCUMENT TYPE STRATA =====
def first_nonempty(row, candidates, default=""):
    for field in candidates:
        if field in row.index:
            value = row[field]
            if pd.notna(value) and str(value).strip():
                return str(value).strip()
    return default

def parse_year(value):
    match = re.search(r"\b(19\d{2}|20\d{2})\b", str(value or ""))
    return int(match.group(1)) if match else None

def infer_document_type(row):
    supplied = first_nonempty(row, TYPE_FIELDS)
    if supplied and supplied.lower() not in {"nan", "none", "unknown", "null"}:
        return supplied[:80]
    s = (first_nonempty(row, TITLE_FIELDS) + " " + first_nonempty(row, URL_FIELDS)).lower()
    rules = [
        ("FOMC / Monetary Policy", ["fomc", "federal open market", "monetary policy", "target range for the federal funds"]),
        ("Speeches / Remarks", ["speech", "remarks", "address", "keynote"]),
        ("Testimony", ["testimony", "testifies"]),
        ("Research / Reports", ["research", "working paper", "survey", "report"]),
        ("Regulation / Supervision", ["supervision", "regulation", "banking", "capital rule", "stress test"]),
        ("Financial Stability", ["financial stability", "financial markets", "liquidity"]),
        ("Payments / Operations", ["payment", "fednow", "currency"]),
    ]
    for label, words in rules:
        if any(word in s for word in words):
            return label
    return "General / Other"

df["_title"] = df.apply(lambda r: first_nonempty(r, TITLE_FIELDS, "(untitled)"), axis=1)
df["_url"] = df.apply(lambda r: first_nonempty(r, URL_FIELDS), axis=1)
df["_date_raw"] = df.apply(lambda r: first_nonempty(r, DATE_FIELDS), axis=1)
df["_year"] = df["_date_raw"].apply(parse_year)
df["_document_type"] = df.apply(infer_document_type, axis=1)
df["_review_text"] = df.apply(lambda r: first_nonempty(r, TEXT_FIELDS), axis=1)

def is_archive_candidate(row):
    flag = str(row.get("archive_type", "")).strip().lower()
    title, url = row["_title"].lower(), row["_url"].lower()
    return flag in {"false", "0", "archive", "archive_page", "index"} or ("/archives/" in url and not row["_date_raw"]) or title == "press releases"

df["_archive_candidate"] = df.apply(is_archive_candidate, axis=1)
frame = df.loc[~df["_archive_candidate"]].copy()
frame["_year_stratum"] = frame["_year"].apply(lambda x: str(int(x)) if pd.notna(x) else "Unknown year")
frame["_type_stratum"] = frame["_document_type"].fillna("General / Other").astype(str)
frame["_stratum"] = frame["_year_stratum"] + " | " + frame["_type_stratum"]

print("Total records:", len(df))
print("Archive candidates excluded:", int(df["_archive_candidate"].sum()))
print("Sampling frame:", len(frame))
display(frame.groupby(["_year_stratum", "_type_stratum"]).size().reset_index(name="population_n"))

Total records: 4629
Archive candidates excluded: 0
Sampling frame: 4629


,_year_stratum,_type_stratum,population_n
0,2006,general,197
1,2007,general,191
2,2008,general,337
3,2009,general,400
4,2010,general,400
5,2011,general,236
6,2012,general,209
7,2013,general,216
8,2014,general,253
9,2015,general,230


In [16]:
# ===== STRATIFIED SAMPLE (proportional allocation) =====
N = len(frame)
n = min(int(SAMPLE_SIZE), N)
if N == 0:
    raise ValueError("Sampling frame ว่าง")
if n < SAMPLE_SIZE:
    print(f"Warning: population มี {N}; สุ่มได้ {n} records")

sizes = frame["_stratum"].value_counts().sort_index()
k = len(sizes)

if k <= n:
    allocation = pd.Series(1, index=sizes.index, dtype=int)
    remaining = n - k
    if remaining:
        ideal = sizes / N * remaining
        floors = np.floor(ideal).astype(int)
        allocation = allocation.add(floors, fill_value=0).astype(int)
        left = n - int(allocation.sum())
        frac = (ideal - floors).sort_values(ascending=False, kind="stable")
        for st in frac.index[:left]:
            allocation.loc[st] += 1
else:
    ideal = sizes / N * n
    allocation = np.floor(ideal).astype(int)
    left = n - int(allocation.sum())
    frac = (ideal - allocation).sort_values(ascending=False, kind="stable")
    for st in frac.index[:left]:
        allocation.loc[st] += 1

allocation = allocation.clip(upper=sizes)
while int(allocation.sum()) < n:
    capacity = sizes - allocation
    eligible = capacity[capacity > 0]
    if eligible.empty:
        break
    st = (eligible / sizes[eligible.index]).sort_values(ascending=False, kind="stable").index[0]
    allocation.loc[st] += 1

rng = np.random.default_rng(RANDOM_SEED)
chosen_indices = []
for st, take in allocation.items():
    candidates = frame.index[frame["_stratum"] == st].to_numpy()
    if take:
        chosen_indices.extend(rng.choice(candidates, size=int(take), replace=False).tolist())

sample = frame.loc[chosen_indices].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
sample["record_id"] = sample.apply(
    lambda r: hashlib.sha256((r["_url"] or f"{r['_title']}|{r['_date_raw']}").encode("utf-8")).hexdigest()[:16],
    axis=1
)
print(f"Sampled {len(sample)} / {N} records across {k} strata; seed={RANDOM_SEED}")
display(pd.DataFrame({"stratum": sizes.index, "population_n": sizes.values,
                      "sample_n": allocation.reindex(sizes.index).fillna(0).astype(int).values}))

Sampled 200 / 4629 records across 28 strata; seed=20261026


,stratum,population_n,sample_n
0,2006 | general,197,8
1,2007 | general,191,8
2,2008 | general,337,14
3,2009 | general,400,16
4,2010 | general,400,16
5,2011 | general,236,10
6,2012 | general,209,9
7,2013 | general,216,9
8,2014 | general,253,10
9,2015 | general,230,10


In [17]:
# ===== IMPORT THAILLM CLIENT (OpenAI-compatible) =====
# ThaiLLM เปิด endpoint แบบ OpenAI-compatible จึงใช้ openai package ตรง ๆ
# หากยังไม่ได้ติดตั้ง ให้รันใน terminal: pip install openai python-dotenv tqdm
# หรือ uncomment บรรทัดด้านล่างใน Jupyter:
# %pip install openai python-dotenv tqdm

try:
    from openai import OpenAI
except ImportError as exc:
    raise ImportError("ยังไม่มี openai package — ติดตั้งด้วย %pip install openai แล้วรัน Cell นี้ใหม่") from exc

# สร้าง client 1 ตัวต่อ 1 API key (7 keys) เพื่อประมวลผลแบบขนาน
clients = [
    OpenAI(base_url=THAILLM_BASE_URL, api_key=key)
    for key in THAILLM_API_KEYS
]
num_keys = len(clients)

def get_llm_index(record_id):
    '''กระจาย record ลง client แต่ละตัวด้วย hash ของ record_id (mod N)'''
    hash_value = int(hashlib.md5(str(record_id).encode("utf-8")).hexdigest(), 16)
    return hash_value % max(1, num_keys)

print(f"ThaiLLM clients initialized: {num_keys} instances @ {THAILLM_BASE_URL}")


ThaiLLM clients initialized: 7 instances @ http://thaillm.or.th/api/v1


In [ ]:
# ===== LLM CLASSIFICATION PROMPT v2 + STRICT JSON PARSER =====
SYSTEM_PROMPT = '''
FED DOCUMENT RELEVANCE AUDITOR — GOLD RESEARCH v2

1. ROLE
You are a conservative document-relevance auditor for a quantitative gold-market research corpus.
Your task is to classify the actual content of a Federal Reserve document for potential use in gold-market research and macroeconomic context retrieval.

You are NOT predicting gold prices.
You are NOT deciding whether an event caused gold prices to move.
You are NOT assuming that every Federal Reserve document is economically important.
You are NOT generating missing facts or inferring the undisclosed contents of a document.

Classify only what is supported by the supplied document text and metadata.

2. INPUT
You may receive:
- record_id
- title
- publicationDate
- url
- document_type or source_type
- article_text

Treat article_text as the primary evidence source.
Use title and metadata to identify the document and its apparent type, but do not use the title alone to claim that the body contains information it does not contain.

If article_text is empty, truncated, inaccessible, or only contains a notice that another document is available, explicitly account for this limitation.
Do not assume that linked PDFs, meeting minutes, reports, tables, or appendices are included unless their contents are present in article_text.

3. CLASSIFICATION LABELS
Choose exactly one primary label.

DIRECT_GOLD
Use only when the supplied document substantively discusses gold, the gold market, gold prices, gold demand/supply, gold investment, or a gold-specific market relationship.
A generic FOMC decision, CPI report, or employment report is NOT DIRECT_GOLD merely because it may affect gold.

MACRO_RELEVANT
Use when the supplied document contains substantive economic information that can serve as a defensible macroeconomic input or context for gold-market research.
Potentially relevant topics include:
- Federal funds rate decisions and monetary policy stance
- Inflation and inflation expectations
- Employment, wages, labor-market conditions
- Economic growth, recession risks, demand conditions
- Treasury yields, real yields, and interest-rate expectations
- US dollar or exchange-rate conditions, when substantively discussed
- Financial stress, systemic banking risk, credit conditions, or broad liquidity conditions
- Central-bank balance-sheet policy and monetary transmission
- Geopolitical or systemic events only when the supplied document substantively addresses their economic or financial implications

The document must contain actual findings, decisions, data, analysis, or substantive policy information.
The mere presence of a keyword is insufficient.

LOW_RELEVANCE
Use when the available document contains no substantive evidence relevant to gold-specific research or the macroeconomic factors listed above.
Examples may include:
- Routine bank branch or merger approvals
- Routine enforcement or compliance notices with no substantive systemic implications
- Administrative schedules and calendars
- Generic announcements that a document will be released, when the substantive document is not included
- Operational or regulatory details with no supported macroeconomic relevance

Do not classify a document as LOW_RELEVANCE solely because it does not mention gold.

UNCERTAIN
Use when the document cannot be reliably classified because essential evidence is missing, truncated, ambiguous, or inaccessible.
Examples:
- A title indicates that meeting minutes or a report may contain substantive information, but the body is unavailable.
- The text is incomplete or corrupted.
- It is unclear whether the supplied text is an announcement or the actual substantive document.
- The document appears potentially relevant, but the available evidence is insufficient to determine its contents.

Do not use UNCERTAIN merely because the eventual effect on gold prices is unknown.

4. EVIDENCE REQUIREMENTS
For DIRECT_GOLD or MACRO_RELEVANT:
Identify at least one specific statement, finding, decision, statistic, or analysis in article_text.
Provide an exact evidence quote copied from article_text.
Explain why that evidence is relevant without claiming a causal effect that the document does not establish.
Identify the economic factor supported by the evidence.
Describe the relevance pathway in a short, defensible chain.
State the key limitation.

Do not fabricate quotations, statistics, policy decisions, or document contents.
If no suitable supporting quote exists, do not invent one. Reconsider the label or use UNCERTAIN if the text is insufficient.

Evidence quotes must be verbatim excerpts from the supplied text. Do not paraphrase inside the quote field.
Quote verification will be performed by a separate program. Do not claim that a quote has been programmatically verified.

5. RELEVANCE PATHWAY RULES
A relevance pathway must be supported by the document.

Acceptable examples:
- FOMC target-rate decision → monetary policy stance → context for interest-rate expectations relevant to gold analysis.
- CPI inflation data → inflation conditions → context for real-rate and inflation-expectation analysis.
- Employment and wage data → labor-market conditions → context for Federal Reserve policy expectations.
- Treasury yield data explicitly reported in the document → yield conditions → context for gold-market research.
- Evidence of broad systemic banking stress → financial-stability conditions → potential safe-haven context, if supported by the text.

Do not assert that the document proves gold prices will rise or fall.
Do not assume:
- a facility auction rate is a Treasury yield or real interest rate;
- a bank-specific regulatory action proves systemic financial stress;
- routine payment-system policy proves a change in broad liquidity;
- a regulatory proposal has already been implemented;
- an announcement contains the contents of the report or minutes it announces.

If the pathway requires multiple unsupported assumptions, classify conservatively and explain the limitation.
An empty relevance pathway is acceptable for LOW_RELEVANCE or UNCERTAIN.

6. DOCUMENT-TYPE HANDLING
Distinguish the following:
- Substantive policy statement versus announcement of a statement
- Actual meeting minutes versus notice that minutes are available
- Economic data release versus administrative announcement about a release
- Published findings versus proposed rules
- System-wide evidence versus an isolated institution-specific event

Classify the supplied text, not an assumed document behind a link.
If the text is only an announcement but the announced material is not included, do not infer its contents.
Use UNCERTAIN when the unavailable content is necessary to determine relevance and the title suggests potentially substantive material.

7. CONFIDENCE
Return a numeric confidence between 0.0 and 1.0.
Confidence means confidence in the classification given the available evidence, not confidence that the document will predict gold prices.

Use lower confidence when:
- the text is incomplete;
- the document type is ambiguous;
- the relevance pathway is indirect;
- key evidence is missing.

High confidence requires clear supporting or excluding evidence in the supplied text.
Do not assign high confidence solely because the title looks familiar or the document type is common.

8. REVIEW POLICY
Set llm_review_required to true if any of the following applies:
- Label is UNCERTAIN.
- Article text is missing, truncated, or appears to omit the substantive document.
- A potentially relevant document is classified LOW_RELEVANCE despite an ambiguous or incomplete body.
- The relevance pathway relies on indirect assumptions.
- The evidence quote is missing for DIRECT_GOLD or MACRO_RELEVANT.
- The distinction between a substantive document and an administrative announcement is unclear.
- Confidence is below 0.75.
- The content appears to concern systemic financial stress, monetary policy, inflation, employment, or yields but its relevance is unclear.

Do not set review_required to false simply because the classification seems plausible.

9. OUTPUT FORMAT
Return exactly one valid JSON object and no Markdown fences or surrounding commentary.

Use exactly these fields and types:
{
"record_id": "string",
"llm_primary_label": "DIRECT_GOLD | MACRO_RELEVANT | LOW_RELEVANCE | UNCERTAIN",
"llm_confidence": 0.0,
"llm_document_type": "string",
"llm_main_topic": "string",
"llm_evidence_quote": "string",
"llm_economic_factor": ["string"],
"llm_relevance_pathway": "string",
"llm_relevance_reason": "string",
"llm_limitations": ["string"],
"llm_review_required": true
}

Schema requirements:
- llm_confidence must be a JSON number.
- llm_economic_factor must be a JSON array of strings, never a string containing serialized JSON.
- llm_limitations must be a JSON array of strings, never a string containing serialized JSON.
- llm_review_required must be a JSON boolean.
- Use [] when no economic factors or limitations can be identified.
- Use an empty string when a text field has no supported value.
- Preserve record_id exactly as supplied.
- Do not add extra fields.
- Do not wrap the JSON object in a list unless the caller explicitly requests a list.
- Do not include comments or trailing commas.

10. FINAL SELF-CHECK
Before returning the JSON object, check:
Does the label match the definitions exactly?
Does every relevance claim have support in article_text?
Is DIRECT_GOLD reserved for substantive gold-specific content?
Is MACRO_RELEVANT based on substantive macroeconomic evidence rather than keywords?
Have you avoided treating an announcement as the contents of an unavailable document?
Is the quote copied verbatim from the supplied text?
Are the JSON field types correct?
Does the review flag reflect uncertainty and evidence quality?

Return only the JSON object.
'''

def parse_llm_json(raw):
    '''ดึง JSON object แรกที่ใช้งานได้ออกจากข้อความ LLM'''
    text = str(raw or "").strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if fence:
        text = fence.group(1).strip()
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            obj, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict) and "llm_primary_label" in obj:
            return obj
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end > start:
        try:
            obj = json.loads(text[start:end + 1])
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            pass
    raise ValueError(f"ไม่พบ JSON object ที่ใช้งานได้ใน output: {text[:200]!r}")

def classify_document(row, client):
    title = str(row["_title"])[:500]
    body = str(row["_review_text"] or "")[:MAX_TEXT_CHARS]
    if not body.strip():
        body = "[No body text available in Clean Master]"
    
    user_payload = {
        "record_id": str(row["record_id"]),
        "title": title,
        "publicationDate": str(row["_date_raw"]),
        "url": str(row["_url"]),
        "document_type": str(row["_document_type"]),
        "article_text": body,
    }
    
    response = client.chat.completions.create(
        model=MODEL,
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
        ],
    )
    raw = response.choices[0].message.content
    result = parse_llm_json(raw)
    
    # Validate and normalize output
    primary_label = result.get("llm_primary_label", "UNCERTAIN")
    if primary_label not in LABELS:
        primary_label = "UNCERTAIN"
    
    confidence = result.get("llm_confidence", None)
    try:
        confidence = float(confidence) if confidence is not None else None
        if confidence is not None:
            confidence = min(1.0, max(0.0, confidence))
    except (TypeError, ValueError):
        confidence = None
    
    evidence_quote = str(result.get("llm_evidence_quote", "") or "")
    
    # CRITICAL: Ensure arrays are not serialized strings
    economic_factor = result.get("llm_economic_factor", [])
    if isinstance(economic_factor, str):
        raise ValueError("llm_economic_factor is a string instead of array")
    if not isinstance(economic_factor, list):
        economic_factor = []
    
    limitations = result.get("llm_limitations", [])
    if isinstance(limitations, str):
        raise ValueError("llm_limitations is a string instead of array")
    if not isinstance(limitations, list):
        limitations = []
    
    review_required = result.get("llm_review_required", False)
    if not isinstance(review_required, bool):
        review_required = False
    
    return {
        "llm_primary_label": primary_label,
        "llm_confidence": confidence,
        "llm_document_type": str(result.get("llm_document_type", "")),
        "llm_main_topic": str(result.get("llm_main_topic", "")),
        "llm_evidence_quote": evidence_quote,
        "llm_economic_factor": economic_factor,
        "llm_relevance_pathway": str(result.get("llm_relevance_pathway", "")),
        "llm_relevance_reason": str(result.get("llm_relevance_reason", "")),
        "llm_limitations": limitations,
        "llm_review_required": review_required,
    }

print("Prompt v2 and parser defined")

In [ ]:
# ===== STAGE 3: DETERMINISTIC VALIDATION (Updated with Canonical Schema) =====
import unicodedata

def normalize_text(text):
    """Normalize text for quote verification (NFKC + whitespace normalization)"""
    text = unicodedata.normalize("NFKC", text or "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def verify_quote(article_text, quote):
    """Verify if quote exists in article text after normalization"""
    quote = normalize_text(quote)
    article = normalize_text(article_text)
    
    if not quote:
        return {
            "quote_verified": False,
            "verification_status": "EMPTY_QUOTE"
        }
    
    found = quote in article
    return {
        "quote_verified": found,
        "verification_status": "EXACT_NORMALIZED_MATCH" if found else "QUOTE_NOT_FOUND"
    }

def validate_against_canonical_schema(obj):
    """Validate object against canonical schema"""
    validation = {
        "schema_valid": True,
        "schema_errors": [],
        "quarantined": False
    }
    
    # Check required fields
    for field in CANONICAL_SCHEMA["required"]:
        if field not in obj or obj[field] is None:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"Missing required field: {field}")
    
    # Check llm_primary_label enum
    if "llm_primary_label" in obj:
        if obj["llm_primary_label"] not in CANONICAL_SCHEMA["properties"]["llm_primary_label"]["enum"]:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"Invalid llm_primary_label: {obj['llm_primary_label']}")
    
    # Check llm_confidence bounds
    if "llm_confidence" in obj and obj["llm_confidence"] is not None:
        confidence = obj["llm_confidence"]
        if not isinstance(confidence, (int, float)):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_confidence must be number, got {type(confidence)}")
        elif confidence < 0.0 or confidence > 1.0:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_confidence out of bounds: {confidence}")
    
    # CRITICAL: Check that arrays are not serialized strings
    if "llm_economic_factor" in obj:
        if isinstance(obj["llm_economic_factor"], str):
            validation["schema_valid"] = False
            validation["schema_errors"].append("llm_economic_factor is string instead of array")
            validation["quarantined"] = True
        elif not isinstance(obj["llm_economic_factor"], list):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_economic_factor not array: {type(obj['llm_economic_factor'])}")
    
    if "llm_limitations" in obj:
        if isinstance(obj["llm_limitations"], str):
            validation["schema_valid"] = False
            validation["schema_errors"].append("llm_limitations is string instead of array")
            validation["quarantined"] = True
        elif not isinstance(obj["llm_limitations"], list):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_limitations not array: {type(obj['llm_limitations'])}")
    
    # Check llm_review_required is boolean
    if "llm_review_required" in obj:
        if not isinstance(obj["llm_review_required"], bool):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_review_required not boolean: {type(obj['llm_review_required'])}")
    
    # Check string fields are not empty (except llm_evidence_quote which can be empty)
    for field in ["record_id", "llm_document_type", "llm_main_topic", "llm_relevance_reason"]:
        if field in obj:
            value = obj[field]
            if value is None or (isinstance(value, str) and not value.strip()):
                validation["schema_valid"] = False
                validation["schema_errors"].append(f"Field {field} is empty or None")
    
    return validation

def improved_validate_schema(parsed_result, body_text):
    """Improved validation using canonical schema"""
    # First validate against canonical schema
    schema_validation = validate_against_canonical_schema(parsed_result)
    
    validation = {
        "schema_valid": schema_validation["schema_valid"],
        "schema_errors": schema_validation["schema_errors"],
        "quote_verification": None,
        "consistency_flags": []
    }
    
    # If schema is invalid, quarantine immediately
    if schema_validation["quarantined"]:
        validation["consistency_flags"].append("SCHEMA_INVALID")
        return validation
    
    # Quote verification
    evidence_quote = parsed_result.get("llm_evidence_quote", "")
    if evidence_quote:
        validation["quote_verification"] = verify_quote(body_text, evidence_quote)
        if not validation["quote_verification"]["quote_verified"]:
            validation["consistency_flags"].append("QUOTE_NOT_FOUND")
    else:
        validation["consistency_flags"].append("EMPTY_QUOTE")
    
    # Rule-based consistency checks (stricter)
    primary_label = parsed_result.get("llm_primary_label")
    confidence = parsed_result.get("llm_confidence")
    economic_factor = parsed_result.get("llm_economic_factor", [])
    
    if primary_label == "DIRECT_GOLD":
        # DIRECT_GOLD requires gold-specific evidence
        if not evidence_quote:
            validation["consistency_flags"].append("DIRECT_GOLD_NO_EVIDENCE")
        validation["consistency_flags"].append("DIRECT_GOLD_REQUIRES_REVIEW")
    
    if primary_label == "MACRO_RELEVANT":
        # MACRO_RELEVANT requires evidence AND economic factors
        if not evidence_quote:
            validation["consistency_flags"].append("MACRO_RELEVANT_NO_EVIDENCE")
        if not economic_factor or len(economic_factor) == 0:
            validation["consistency_flags"].append("MACRO_RELEVANT_NO_FACTORS")
    
    if primary_label == "UNCERTAIN":
        validation["consistency_flags"].append("UNCERTAIN_REQUIRES_REVIEW")
    
    if confidence is not None and confidence < 0.75:
        validation["consistency_flags"].append("LOW_CONFIDENCE")
    
    if not body_text.strip() or body_text == "[No body text available in Clean Master]":
        validation["consistency_flags"].append("MISSING_BODY_TEXT")
    
    return validation

print("Validation functions defined (using canonical schema)")

In [ ]:
# ===== STAGE 2 + 3: LLM CLASSIFICATION + DETERMINISTIC VALIDATION (Updated) =====
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
progress_path = OUTPUT_DIR / "llm_progress_v2.jsonl"  # Use v2 to avoid conflicts
errors_path = OUTPUT_DIR / "llm_errors_v2.jsonl"
results_by_id = {}
write_lock = Lock()

# Resume from prior progress file if it exists
if progress_path.exists():
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    item = json.loads(line)
                    results_by_id[item["record_id"]] = item
                except Exception:
                    pass
    print("Resuming:", len(results_by_id), "prior results found")
else:
    print("Starting fresh run (v2)")

pending = sample.loc[~sample["record_id"].isin(results_by_id)].copy()
print("Pending LLM classifications:", len(pending), "| workers:", num_keys)

def process_record(row):
    record_id = row["record_id"]
    client = clients[get_llm_index(record_id)]
    result = None
    last_error = None
    
    # Stage 2: LLM Classification
    for attempt in range(1, MAX_RETRIES + 1):
        time.sleep(THAILLM_DELAY)
        try:
            result = classify_document(row, client)
            break
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(min(2 ** attempt, 20))
    
    if result is None:
        with write_lock:
            with errors_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(
                    {"record_id": record_id, "title": row["_title"], "error": last_error},
                    ensure_ascii=False) + "\n")
        result = {
            "llm_primary_label": "UNCERTAIN", "llm_confidence": None,
            "llm_document_type": "", "llm_main_topic": "",
            "llm_evidence_quote": "", "llm_economic_factor": [],
            "llm_relevance_pathway": "", "llm_relevance_reason": f"API error after retries: {last_error}",
            "llm_limitations": [], "llm_review_required": True,
            "llm_error": last_error,
        }
    
    # Stage 3: Deterministic Validation (using improved validation)
    body_text = str(row["_review_text"] or "")[:MAX_TEXT_CHARS]
    if not body_text.strip():
        body_text = "[No body text available in Clean Master]"
    
    validation = improved_validate_schema(result, body_text)
    
    # Merge validation results
    result["validation_schema_valid"] = validation["schema_valid"]
    result["validation_schema_errors"] = validation["schema_errors"]
    result["validation_quote_verified"] = validation["quote_verification"]["quote_verified"] if validation["quote_verification"] else None
    result["validation_quote_status"] = validation["quote_verification"]["verification_status"] if validation["quote_verification"] else None
    result["validation_consistency_flags"] = validation["consistency_flags"]
    
    # Auto-set review_required based on validation flags
    if validation["consistency_flags"]:
        result["llm_review_required"] = True
    
    # Store raw output for audit trail
    result["input_hash"] = hashlib.sha256(body_text.encode("utf-8")).hexdigest()[:16]
    result["prompt_version"] = "fed_gold_relevance_v2"
    result["model"] = MODEL
    
    item = {"record_id": record_id, **result}
    with write_lock:
        with progress_path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        results_by_id[record_id] = item
    return item

if len(pending):
    with ThreadPoolExecutor(max_workers=num_keys) as executor:
        futures = [executor.submit(process_record, row) for _, row in pending.iterrows()]
        for _ in tqdm(as_completed(futures), total=len(futures), desc="ThaiLLM classify + validate v2"):
            pass

print(f"LLM pass complete. Total results={len(results_by_id)}")

In [ ]:
# ===== STAGE 1 + 4: MERGE WITH CLEAN MASTER + EXPORT FOR HUMAN AUDIT (v2) =====
# Check if LLM classification has been run
if not results_by_id:
    print("ERROR: No LLM results found. Please run Cell 8 (Stage 2+3) first to classify documents.")
    print("This cell requires LLM classification results to merge with the sample.")
else:
    results_df = pd.DataFrame(list(results_by_id.values()))
    merged = sample.merge(results_df, on="record_id", how="left", validate="one_to_one")

    review_df = pd.DataFrame({
        "record_id": merged["record_id"],
        "published_date": merged["_date_raw"],
        "year": merged["_year_stratum"],
        "document_type": merged["_type_stratum"],
        "title": merged["_title"],
        "url": merged["_url"],
        "article_text": merged["_review_text"],
        "llm_primary_label": merged.get("llm_primary_label", pd.Series(index=merged.index, dtype=object)),
        "llm_confidence": merged.get("llm_confidence", pd.Series(index=merged.index, dtype=object)),
        "llm_document_type": merged.get("llm_document_type", pd.Series(index=merged.index, dtype=object)),
        "llm_main_topic": merged.get("llm_main_topic", pd.Series(index=merged.index, dtype=object)),
        "llm_evidence_quote": merged.get("llm_evidence_quote", pd.Series(index=merged.index, dtype=object)),
        "llm_economic_factor": merged.get("llm_economic_factor", pd.Series(index=merged.index, dtype=object)),
        "llm_relevance_pathway": merged.get("llm_relevance_pathway", pd.Series(index=merged.index, dtype=object)),
        "llm_relevance_reason": merged.get("llm_relevance_reason", pd.Series(index=merged.index, dtype=object)),
        "llm_limitations": merged.get("llm_limitations", pd.Series(index=merged.index, dtype=object)),
        "llm_review_required": merged.get("llm_review_required", pd.Series(index=merged.index, dtype=object)),
        "validation_schema_valid": merged.get("validation_schema_valid", pd.Series(index=merged.index, dtype=object)),
        "validation_schema_errors": merged.get("validation_schema_errors", pd.Series(index=merged.index, dtype=object)),
        "validation_quote_verified": merged.get("validation_quote_verified", pd.Series(index=merged.index, dtype=object)),
        "validation_quote_status": merged.get("validation_quote_status", pd.Series(index=merged.index, dtype=object)),
        "validation_consistency_flags": merged.get("validation_consistency_flags", pd.Series(index=merged.index, dtype=object)),
        "input_hash": merged.get("input_hash", pd.Series(index=merged.index, dtype=object)),
        "prompt_version": merged.get("prompt_version", pd.Series(index=merged.index, dtype=object)),
        "model": merged.get("model", pd.Series(index=merged.index, dtype=object)),
        "human_label": "",
        "human_confidence": "",
        "human_evidence_quote": "",
        "human_reason": "",
        "source_complete": "",
        "adjudication_required": "",
        "sampling_stratum": merged["_stratum"],
        "sampling_seed": RANDOM_SEED,
    })

    review_path = OUTPUT_DIR / "fed_gold_relevance_llm_human_review_v2.csv"
    review_df.to_csv(review_path, index=False, encoding="utf-8-sig")
    print("Review CSV (v2):", review_path.resolve())

    priority_review = review_df[
        (review_df["llm_review_required"] == True) |
        (review_df["llm_primary_label"] == "MACRO_RELEVANT") |
        (review_df["llm_primary_label"] == "DIRECT_GOLD") |
        (review_df["validation_consistency_flags"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False))
    ]
    print(f"Priority review items: {len(priority_review)} / {len(review_df)}")
    display(review_df[["title", "llm_primary_label", "llm_confidence", "validation_quote_verified", "validation_consistency_flags", "llm_review_required"]].head(10))

In [ ]:
# ===== SAVE SAMPLING + METADATA + VALIDATION SUMMARY (v2) =====
sizes = frame["_stratum"].value_counts().sort_index()
strata_metadata = []
for st in sizes.index:
    strata_metadata.append({
        "stratum": st,
        "population_n": int(sizes.loc[st]),
        "sample_n": int(allocation.get(st, 0)),
        "sampling_fraction": float(allocation.get(st, 0) / sizes.loc[st]),
    })

# Define review_path if not already defined
if 'review_path' not in locals():
    review_path = OUTPUT_DIR / "fed_gold_relevance_llm_human_review_v2.csv"

metadata = {
    "project": "Gold Assistant - Federal Reserve relevance audit (v2 + canonical schema)",
    "input_path": str(INPUT_PATH),
    "input_record_count": int(len(df)),
    "archive_candidates_excluded": int(df["_archive_candidate"].sum()),
    "sampling_frame_count": int(len(frame)),
    "requested_sample_size": int(SAMPLE_SIZE),
    "actual_sample_size": int(len(sample)),
    "random_seed": int(RANDOM_SEED),
    "sampling_method": "proportional stratified sampling by year × document_type",
    "strata_count": int(len(sizes)),
    "llm_provider": PROVIDER,
    "thaillm_base_url": THAILLM_BASE_URL,
    "model": MODEL,
    "api_keys_used": int(num_keys),
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "max_text_chars": MAX_TEXT_CHARS,
    "prompt_version": "fed_gold_relevance_v2",
    "labels": LABELS,
    "human_review_csv": str(review_path),
    "progress_file": str(progress_path),
    "input_file_modified": False,
    "strata": strata_metadata,
    "validation_enabled": True,
    "quote_verification_enabled": True,
    "rule_based_consistency_checks": True,
    "canonical_schema": "fed_gold_relevance_v2",
}
with (OUTPUT_DIR / "sampling_and_llm_metadata_v2.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Label summary - only if review_df exists
if 'review_df' in locals():
    summary = review_df["llm_primary_label"].value_counts(dropna=False).rename_axis("llm_primary_label").reset_index(name="count")
    summary["share"] = summary["count"] / max(len(review_df), 1)
    summary.to_csv(OUTPUT_DIR / "llm_label_summary_v2.csv", index=False, encoding="utf-8-sig")
    display(summary)
else:
    print("Warning: review_df not found. Run Cell 9 first to generate review data.")

# Validation summary - only if review_df exists
if 'review_df' in locals():
    validation_summary = {
        "total_records": len(review_df),
        "schema_valid": int(review_df["validation_schema_valid"].sum()),
        "schema_invalid": int((~review_df["validation_schema_valid"]).sum()),
        "quote_verified": int(review_df["validation_quote_verified"].sum()),
        "quote_not_found": int((review_df["validation_quote_status"] == "QUOTE_NOT_FOUND").sum()),
        "empty_quote": int((review_df["validation_quote_status"] == "EMPTY_QUOTE").sum()),
        "review_required": int(review_df["llm_review_required"].sum()),
        "has_consistency_flags": int(review_df["validation_consistency_flags"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()),
    }
    with (OUTPUT_DIR / "validation_summary_v2.json").open("w", encoding="utf-8") as f:
        json.dump(validation_summary, f, ensure_ascii=False, indent=2)
    print("\nValidation summary (v2):")
    print(json.dumps(validation_summary, indent=2))
else:
    print("Warning: Cannot generate validation summary. Run Cell 9 first.")

print("Metadata saved (v2).")

In [ ]:
# ===== PHASE B: IMPROVED VALIDATION WITH CANONICAL SCHEMA =====
def validate_against_canonical_schema(obj: Dict[str, Any]) -> Dict[str, Any]:
    """Validate object against canonical schema"""
    validation = {
        "schema_valid": True,
        "schema_errors": [],
        "quarantined": False
    }
    
    # Check required fields
    for field in CANONICAL_SCHEMA["required"]:
        if field not in obj or obj[field] is None:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"Missing required field: {field}")
    
    # Check llm_primary_label enum
    if "llm_primary_label" in obj:
        if obj["llm_primary_label"] not in CANONICAL_SCHEMA["properties"]["llm_primary_label"]["enum"]:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"Invalid llm_primary_label: {obj['llm_primary_label']}")
    
    # Check llm_confidence bounds
    if "llm_confidence" in obj and obj["llm_confidence"] is not None:
        confidence = obj["llm_confidence"]
        if not isinstance(confidence, (int, float)):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_confidence must be number, got {type(confidence)}")
        elif confidence < 0.0 or confidence > 1.0:
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_confidence out of bounds: {confidence}")
    
    # CRITICAL: Check that arrays are not serialized strings
    if "llm_economic_factor" in obj:
        if isinstance(obj["llm_economic_factor"], str):
            validation["schema_valid"] = False
            validation["schema_errors"].append("llm_economic_factor is string instead of array")
            validation["quarantined"] = True
        elif not isinstance(obj["llm_economic_factor"], list):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_economic_factor not array: {type(obj['llm_economic_factor'])}")
    
    if "llm_limitations" in obj:
        if isinstance(obj["llm_limitations"], str):
            validation["schema_valid"] = False
            validation["schema_errors"].append("llm_limitations is string instead of array")
            validation["quarantined"] = True
        elif not isinstance(obj["llm_limitations"], list):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_limitations not array: {type(obj['llm_limitations'])}")
    
    # Check llm_review_required is boolean
    if "llm_review_required" in obj:
        if not isinstance(obj["llm_review_required"], bool):
            validation["schema_valid"] = False
            validation["schema_errors"].append(f"llm_review_required not boolean: {type(obj['llm_review_required'])}")
    
    # Check string fields are not empty (except llm_evidence_quote which can be empty)
    for field in ["record_id", "llm_document_type", "llm_main_topic", "llm_relevance_reason"]:
        if field in obj:
            value = obj[field]
            if value is None or (isinstance(value, str) and not value.strip()):
                validation["schema_valid"] = False
                validation["schema_errors"].append(f"Field {field} is empty or None")
    
    return validation

def improved_validate_schema(parsed_result: Dict[str, Any], body_text: str) -> Dict[str, Any]:
    """Improved validation using canonical schema"""
    # First validate against canonical schema
    schema_validation = validate_against_canonical_schema(parsed_result)
    
    validation = {
        "schema_valid": schema_validation["schema_valid"],
        "schema_errors": schema_validation["schema_errors"],
        "quote_verification": None,
        "consistency_flags": []
    }
    
    # If schema is invalid, quarantine immediately
    if schema_validation["quarantined"]:
        validation["consistency_flags"].append("SCHEMA_INVALID")
        return validation
    
    # Quote verification
    evidence_quote = parsed_result.get("llm_evidence_quote", "")
    if evidence_quote:
        validation["quote_verification"] = verify_quote(body_text, evidence_quote)
        if not validation["quote_verification"]["quote_verified"]:
            validation["consistency_flags"].append("QUOTE_NOT_FOUND")
    else:
        validation["consistency_flags"].append("EMPTY_QUOTE")
    
    # Rule-based consistency checks (stricter)
    primary_label = parsed_result.get("llm_primary_label")
    confidence = parsed_result.get("llm_confidence")
    economic_factor = parsed_result.get("llm_economic_factor", [])
    
    if primary_label == "DIRECT_GOLD":
        # DIRECT_GOLD requires gold-specific evidence
        if not evidence_quote:
            validation["consistency_flags"].append("DIRECT_GOLD_NO_EVIDENCE")
        validation["consistency_flags"].append("DIRECT_GOLD_REQUIRES_REVIEW")
    
    if primary_label == "MACRO_RELEVANT":
        # MACRO_RELEVANT requires evidence AND economic factors
        if not evidence_quote:
            validation["consistency_flags"].append("MACRO_RELEVANT_NO_EVIDENCE")
        if not economic_factor or len(economic_factor) == 0:
            validation["consistency_flags"].append("MACRO_RELEVANT_NO_FACTORS")
    
    if primary_label == "UNCERTAIN":
        validation["consistency_flags"].append("UNCERTAIN_REQUIRES_REVIEW")
    
    if confidence is not None and confidence < 0.75:
        validation["consistency_flags"].append("LOW_CONFIDENCE")
    
    if not body_text.strip() or body_text == "[No body text available in Clean Master]":
        validation["consistency_flags"].append("MISSING_BODY_TEXT")
    
    return validation

# Save canonical schema to file
with (OUTPUT_DIR / "canonical_schema.json").open("w", encoding="utf-8") as f:
    json.dump(CANONICAL_SCHEMA, f, ensure_ascii=False, indent=2)

print("Improved validation functions defined")
print(f"Canonical schema saved to: {OUTPUT_DIR / 'canonical_schema.json'}")

In [ ]:
# ===== PHASE B: CANONICAL JSON SCHEMA =====
import json
from typing import Dict, Any, List

CANONICAL_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "Fed Gold Relevance Classification Output",
    "description": "Canonical schema for Fed document relevance classification for gold market research",
    "type": "object",
    "required": [
        "record_id",
        "llm_primary_label",
        "llm_confidence",
        "llm_document_type",
        "llm_main_topic",
        "llm_evidence_quote",
        "llm_economic_factor",
        "llm_relevance_pathway",
        "llm_relevance_reason",
        "llm_limitations",
        "llm_review_required",
    ],
    "properties": {
        "record_id": {
            "type": "string",
            "minLength": 1,
            "description": "Unique identifier copied exactly from input"
        },
        "llm_primary_label": {
            "type": "string",
            "enum": ["DIRECT_GOLD", "MACRO_RELEVANT", "LOW_RELEVANCE", "UNCERTAIN"],
            "description": "Primary classification label"
        },
        "llm_confidence": {
            "type": "number",
            "minimum": 0.0,
            "maximum": 1.0,
            "description": "Model self-assessed confidence (0-1)"
        },
        "llm_document_type": {
            "type": "string",
            "minLength": 1,
            "description": "Document type classification"
        },
        "llm_main_topic": {
            "type": "string",
            "minLength": 1,
            "description": "Main subject of the document"
        },
        "llm_evidence_quote": {
            "type": "string",
            "description": "Exact quotation from source text (may be empty for LOW_RELEVANCE or UNCERTAIN)"
        },
        "llm_economic_factor": {
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Economic factors identified (must be array, not string)"
        },
        "llm_relevance_pathway": {
            "type": "string",
            "description": "Evidence -> factor -> research use pathway (may be empty)"
        },
        "llm_relevance_reason": {
            "type": "string",
            "minLength": 1,
            "description": "Explanation of classification"
        },
        "llm_limitations": {
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Identified limitations (must be array, not string)"
        },
        "llm_review_required": {
            "type": "boolean",
            "description": "Whether human review is required"
        },
        # Validation fields (generated by code, not from LLM)
        "validation_schema_valid": {
            "type": "boolean",
            "description": "Whether output passes schema validation"
        },
        "validation_schema_errors": {
            "type": "array",
            "items": {
                "type": "string"
            },
            "description": "Schema validation errors"
        },
        "validation_quote_verified": {
            "type": ["boolean", "null"],
            "description": "Whether quote was found in source text"
        },
        "validation_quote_status": {
            "type": "string",
            "enum": ["EXACT_NORMALIZED_MATCH", "QUOTE_NOT_FOUND", "EMPTY_QUOTE", "SOURCE_TEXT_UNAVAILABLE", "QUOTE_NOT_REQUIRED"],
            "description": "Quote verification status"
        },
        "validation_consistency_flags": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": [
                    "QUOTE_NOT_FOUND",
                    "EMPTY_QUOTE",
                    "DIRECT_GOLD_NO_EVIDENCE",
                    "DIRECT_GOLD_REQUIRES_REVIEW",
                    "MACRO_RELEVANT_NO_EVIDENCE",
                    "MACRO_RELEVANT_NO_FACTORS",
                    "UNCERTAIN_REQUIRES_REVIEW",
                    "LOW_CONFIDENCE",
                    "MISSING_BODY_TEXT",
                    "SCHEMA_INVALID",
                    "SOURCE_TEXT_MISSING",
                    "PATHWAY_UNSUPPORTED",
                    "LABEL_EVIDENCE_CONFLICT",
                    "METADATA_CONFLICT"
                ]
            },
            "description": "Rule-based consistency flags"
        },
        # Audit trail fields
        "input_hash": {
            "type": "string",
            "description": "Hash of input text for audit trail"
        },
        "prompt_version": {
            "type": "string",
            "description": "Prompt version used"
        },
        "model": {
            "type": "string",
            "description": "Model identifier"
        },
    }
}

print("Canonical JSON Schema defined")
print(f"Schema version: fed_gold_relevance_v2")
print(f"Required fields: {len(CANONICAL_SCHEMA['required'])}")
print(f"Total properties: {len(CANONICAL_SCHEMA['properties'])}")

In [ ]:
# ===== PHASE A: BASELINE AUDIT REPORT =====
import json
from datetime import datetime

print("=" * 60)
print("PHASE A: BASELINE AUDIT REPORT")
print("=" * 60)
print(f"Timestamp: {datetime.now().isoformat()}")
print()

# 1. Clean Master Inspection
print("1. CLEAN MASTER INSPECTION")
print("-" * 60)
clean_master_path = INPUT_PATH
if clean_master_path.exists():
    clean_count = 0
    clean_schema = set()
    with clean_master_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                clean_count += 1
                clean_schema.update(obj.keys())
    print(f"   Path: {clean_master_path}")
    print(f"   Total records: {clean_count:,}")
    print(f"   Schema fields: {sorted(clean_schema)}")
else:
    print(f"   ERROR: Clean Master not found at {clean_master_path}")
print()

# 2. Progress File Inspection
print("2. PROGRESS FILE INSPECTION")
print("-" * 60)
if progress_path.exists():
    progress_count = 0
    progress_record_ids = set()
    schema_errors = 0
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                progress_count += 1
                if "record_id" in obj:
                    progress_record_ids.add(obj["record_id"])
                if obj.get("validation_schema_valid") == False:
                    schema_errors += 1
    print(f"   Path: {progress_path}")
    print(f"   Total records: {progress_count}")
    print(f"   Unique record_ids: {len(progress_record_ids)}")
    print(f"   Schema invalid: {schema_errors}")
else:
    print(f"   Progress file not found (first run expected)")
print()

# 3. Error File Inspection
print("3. ERROR FILE INSPECTION")
print("-" * 60)
if errors_path.exists():
    error_count = 0
    error_types = {}
    with errors_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                error_count += 1
                if "error" in obj:
                    error_type = obj["error"].split(":")[0]
                    error_types[error_type] = error_types.get(error_type, 0) + 1
    print(f"   Path: {errors_path}")
    print(f"   Total errors: {error_count}")
    print(f"   Error types: {error_types}")
else:
    print(f"   No errors recorded")
print()

# 4. Label Distribution
print("4. LABEL DISTRIBUTION")
print("-" * 60)
if progress_path.exists():
    label_counts = {}
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                label = obj.get("llm_primary_label", "UNKNOWN")
                label_counts[label] = label_counts.get(label, 0) + 1
    for label, count in sorted(label_counts.items()):
        share = count / progress_count * 100
        print(f"   {label}: {count} ({share:.1f}%)")
print()

# 5. Validation Summary
print("5. VALIDATION SUMMARY")
print("-" * 60)
if progress_path.exists():
    quote_verified = 0
    quote_not_found = 0
    empty_quote = 0
    has_flags = 0
    review_required = 0
    
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                if obj.get("validation_quote_verified") == True:
                    quote_verified += 1
                if obj.get("validation_quote_status") == "QUOTE_NOT_FOUND":
                    quote_not_found += 1
                if obj.get("validation_quote_status") == "EMPTY_QUOTE":
                    empty_quote += 1
                if obj.get("validation_consistency_flags") and len(obj["validation_consistency_flags"]) > 0:
                    has_flags += 1
                if obj.get("llm_review_required") == True:
                    review_required += 1
    
    print(f"   Quote verified: {quote_verified} ({quote_verified/progress_count*100:.1f}%)")
    print(f"   Quote not found: {quote_not_found} ({quote_not_found/progress_count*100:.1f}%)")
    print(f"   Empty quote: {empty_quote} ({empty_quote/progress_count*100:.1f}%)")
    print(f"   Has consistency flags: {has_flags} ({has_flags/progress_count*100:.1f}%)")
    print(f"   Review required: {review_required} ({review_required/progress_count*100:.1f}%)")
print()

# 6. Configuration
print("6. CONFIGURATION")
print("-" * 60)
print(f"   Model: {MODEL}")
print(f"   Provider: {PROVIDER}")
print(f"   Temperature: {TEMPERATURE}")
print(f"   Max tokens: {MAX_TOKENS}")
print(f"   Max text chars: {MAX_TEXT_CHARS}")
print(f"   API keys: {num_keys}")
print(f"   Delay per key: {THAILLM_DELAY}s")
print(f"   Sample size: {SAMPLE_SIZE}")
print(f"   Random seed: {RANDOM_SEED}")
print()

print("=" * 60)
print("PHASE A COMPLETE")
print("=" * 60)

## Review instructions (v2 + Validation)

### Pipeline Overview
This notebook implements a 4-stage pipeline:
1. **Stage 1**: Join LLM output with Clean Master (preserves original text)
2. **Stage 2**: LLM classification with FED DOCUMENT RELEVANCE AUDITOR v2 prompt
3. **Stage 3**: Deterministic validation (schema, quote verification, rule-based checks)
4. **Stage 4**: Human audit with priority queue

### Human Audit Procedure

**Step 1: Priority Review**
Review items marked with `llm_review_required = True` first (18 items from previous run).

**Step 2: MACRO_RELEVANT Review**
Review all `MACRO_RELEVANT` items to verify they contain substantive macroeconomic evidence, not just keyword mentions (21 items from previous run).

**Step 3: DIRECT_GOLD Review**
Review all `DIRECT_GOLD` items to verify they discuss gold explicitly (15 items from previous run).

**Step 4: LOW_RELEVANCE Sample**
Randomly sample 30-50 `LOW_RELEVANCE` items stratified by year and document type to check for false negatives.

**Key Audit Candidates** (from previous analysis):
- `a552359c8ac357e9` — Term Auction Facility results
- `8007a5d28beb4e71` — Wildfire support for affected institutions
- `8d267f19ecbeb65c` — Credit card rule clarification
- `4869260301f61f9c` — Discount rate approval
- `e27584ffed3763fd`, `1d06590d946fa983`, `92de90687f15b64a` — Discount rate meeting minutes

### Review CSV Fields

**LLM Output:**
- `llm_primary_label`: LLM classification
- `llm_confidence`: LLM self-assessed confidence (0-1)
- `llm_evidence_quote`: Evidence quote from article
- `llm_economic_factor`: Identified economic factors (array)
- `llm_relevance_pathway`: Evidence → factor → research use
- `llm_relevance_reason`: Explanation
- `llm_limitations`: Identified limitations (array)

**Validation (Stage 3):**
- `validation_schema_valid`: JSON schema passed?
- `validation_quote_verified`: Quote found in original text?
- `validation_quote_status`: EXACT_NORMALIZED_MATCH | QUOTE_NOT_FOUND | EMPTY_QUOTE
- `validation_consistency_flags`: Rule-based flags (e.g., DIRECT_GOLD_NO_EVIDENCE, LOW_CONFIDENCE)

**Human Audit:**
- `human_label`: Your classification (DIRECT_GOLD | MACRO_RELEVANT | LOW_RELEVANCE | UNCERTAIN)
- `human_confidence`: Your confidence (0-1)
- `human_evidence_quote`: Quote you believe supports the classification
- `human_reason`: Your reasoning
- `source_complete`: Is the article text complete and usable? (YES | NO | PARTIAL)
- `adjudication_required`: Does this need second reviewer? (YES | NO)

### Label Definitions (v2)

**DIRECT_GOLD**
Document substantively discusses gold, gold market, gold prices, gold demand/supply, or gold-specific relationships.
FOMC decisions, CPI reports, or employment reports are NOT DIRECT_GOLD unless they mention gold explicitly.

**MACRO_RELEVANT**
Document contains substantive economic information for gold-market research context:
- Federal funds rate decisions, monetary policy stance
- Inflation and inflation expectations
- Employment, wages, labor-market conditions
- Economic growth, recession risks
- Treasury yields, real yields, interest-rate expectations
- US dollar conditions (when substantive)
- Financial stress, systemic banking risk, credit conditions, liquidity
- Central-bank balance-sheet policy

**LOW_RELEVANCE**
No substantive evidence for gold-specific research or macroeconomic factors:
- Routine bank approvals, administrative announcements
- Enforcement notices without systemic implications
- Announcements of documents without substantive content
- Operational details without macroeconomic relevance

**UNCERTAIN**
Insufficient evidence to classify:
- Title suggests substantive content but body is unavailable
- Text is incomplete or corrupted
- Ambiguous whether text is announcement or substantive document

### Important Limitations

- Prompt v2 is more conservative: FOMC decisions without gold mention are MACRO_RELEVANT, not DIRECT_GOLD
- Schema validation ensures arrays are not serialized as strings
- Quote verification is deterministic, not LLM-claimed
- Consistency flags auto-mark items for review based on rules
- LOW_RELEVANCE does not mean the document has no economic value—it means it has little value for this specific RAG use case
- Clean Master is read-only; all outputs are in separate folder

### Evaluation Metrics (After Human Audit)

After completing human review, calculate:
- Precision of DIRECT_GOLD
- Precision and Recall of MACRO_RELEVANT
- Recall of relevant documents (DIRECT_GOLD + MACRO_RELEVANT combined)
- UNCERTAIN rate
- Review required rate
- Quote verification failure rate
- Schema failure rate

**Suggested thresholds:**
- Precision of RAG candidates ≥ 90%
- Recall of human-labeled relevant documents ≥ 90%
- JSON schema valid: 100% after validation/retry
- Quote verification: 100% for quotes used as evidence

### File Locations

- Review CSV: `fed_gold_relevance_llm_human_review.csv`
- Metadata: `sampling_and_llm_metadata.json`
- Validation summary: `validation_summary.json`
- Progress: `llm_progress.jsonl`
- Errors: `llm_errors.jsonl`